# F1 Points-Finish Predictor: Binary Neural Network
**Ryan Lacerna · CAI2300C (Dr. Lee) · Build your own neural network with your own dataset**

Predicts whether a Formula 1 driver will score championship points in a race (1 = yes, 0 = no), using only information known before the race starts.
Built with Dr. Lee's Neural Network Cookbook (02 Binary classifier). Files: [github.com/lacerna5492/f1-neural-network](https://github.com/lacerna5492/f1-neural-network)

## Dataset
My own F1 data (1950 to mid-2026). I combined 8 tables (results, races, drivers, teams, circuits, qualifying and standings) into one table with one row per driver per race: 25,939 rows. The script is `build_table.py` in the repo.

- **Inputs (all known before the race):** season, round, track, team, grid position, field size, driver age and experience, recent driver and team form, and last season's championship standings.
- **Target:** `points_finish` (1 if the driver scored points).
- **Left out on purpose:** finishing position, points, laps, status and race time. These are only known after the race and would leak the answer.

## Step 0 · Processor
Checks that Colab is using the T4 GPU.

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__)
print("Processor:", "GPU " + gpus[0].name if gpus else "CPU only (Runtime → Change runtime type → T4 GPU)")

## Step 1 · Settings
Points the recipe at my dataset and sets `points_finish` as the column to predict. The name columns are dropped because they are labels only.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/lacerna5492/f1-neural-network/main/f1_points_prerace.csv"
TARGET = "points_finish"   # 1 = the driver scored championship points in that race, 0 = no points
PROBLEM = "auto"           # or "regression", "binary", "multiclass"
DROP = ["race_name", "driver_name"]   # human-readable labels only; the stats columns describe the race and driver
SEP = ","                  # use ";" or "\t" if your file isn't comma-separated

## Step 2 · Load the data
Reads the CSV and shows the first rows.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(DATA_URL, sep=SEP)
print(f"{df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Column types, missing values and unique values.

In [ ]:
pd.DataFrame({
    "type": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing %": (df.isna().mean() * 100).round(1),
    "unique values": df.nunique(),
})

## Step 3 · Clean
Drops the listed columns, plus any empty rows, duplicates or ID-like columns.

In [ ]:
df = df.copy()
df.columns = df.columns.str.strip()
log = []

df = df.drop(columns=[c for c in DROP if c in df.columns])
log += [f"dropped {c}: listed in DROP" for c in DROP]

for c in df.select_dtypes(exclude="number").columns:
    s = df[c].astype("string").str.strip().replace({"": pd.NA, "?": pd.NA, "NA": pd.NA, "N/A": pd.NA, "null": pd.NA})
    as_num = pd.to_numeric(s.str.replace(r"[$,%]", "", regex=True), errors="coerce")
    if s.notna().any() and as_num.notna().sum() >= 0.9 * s.notna().sum():
        df[c] = as_num
        log.append(f"converted {c} from text to numbers")
    else:
        df[c] = s.astype(object).where(s.notna(), np.nan)

before = len(df)
df = df.dropna(subset=[TARGET])
log.append(f"dropped {before - len(df)} rows with no {TARGET}")
before = len(df)
df = df.drop_duplicates()
log.append(f"dropped {before - len(df)} duplicate rows")

for c in [c for c in df.columns if c != TARGET]:
    s = df[c]
    if s.isna().mean() > 0.5:
        why = "more than half missing"
    elif s.nunique() <= 1:
        why = "same value in every row"
    elif s.dtype == object and s.nunique() > 0.9 * len(df):
        why = "different in every row (an ID or free text)"
    elif pd.api.types.is_integer_dtype(s) and s.is_unique and (s.sort_values().diff().dropna() == 1).all():
        why = "a row number"
    else:
        continue
    df = df.drop(columns=c)
    log.append(f"dropped {c}: {why}")

print("\n".join(log))
print(f"\nClean table: {df.shape[0]:,} rows × {df.shape[1]} columns")

Only the two name columns are dropped; no rows are lost, because the data was cleaned when I built the table.

## Step 4 · Problem type
The target has two values (0/1), so the recipe picks binary classification.

In [ ]:
y_all = df[TARGET]
X_all = df.drop(columns=TARGET)
n_values = y_all.nunique()
is_number = pd.api.types.is_numeric_dtype(y_all) and not pd.api.types.is_bool_dtype(y_all)

if PROBLEM != "auto":
    problem = PROBLEM
elif not is_number or n_values <= 2 or (pd.api.types.is_integer_dtype(y_all) and n_values <= 10):
    problem = "binary" if n_values == 2 else "multiclass"
else:
    problem = "regression"

print(f"Target '{TARGET}' has {n_values} different values → {problem.upper()}")
if problem != "regression":
    print(y_all.value_counts().to_string())

## Step 5 · Split
Splits the rows 70% training, 15% validation and 15% test.

In [ ]:
from sklearn.model_selection import train_test_split

strat = y_all if problem != "regression" else None
X_train, X_rest, y_train_raw, y_rest = train_test_split(X_all, y_all, test_size=0.3, random_state=42, stratify=strat)
X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=42, stratify=y_rest if strat is not None else None)
print(f"train {len(X_train):,} · validation {len(X_val):,} · test {len(X_test):,}")

## Step 6 · Encode
Scales the number columns and one-hot encodes the team and track. This is fitted on the training rows only, so no test information leaks into training.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

num_cols = X_all.select_dtypes("number").columns.tolist()
cat_cols = [c for c in X_all.columns if c not in num_cols]
prep = ColumnTransformer([
    ("numbers", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("categories", make_pipeline(SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="infrequent_if_exist", max_categories=20, sparse_output=False)), cat_cols),
])
X_tr = prep.fit_transform(X_train).astype("float32")
X_va = prep.transform(X_val).astype("float32")
X_te = prep.transform(X_test).astype("float32")

if problem == "regression":
    y_mean, y_std = y_train_raw.mean(), y_train_raw.std()   # scale the target too; we undo it for the score
    y_tr, y_va, y_te = [((y - y_mean) / y_std).to_numpy("float32") for y in (y_train_raw, y_val_raw, y_test_raw)]
else:
    labels = LabelEncoder().fit(y_all.astype(str))
    y_tr, y_va, y_te = [labels.transform(y.astype(str)) for y in (y_train_raw, y_val_raw, y_test_raw)]
    print("classes:", list(labels.classes_))

print(f"{len(num_cols)} number columns + {len(cat_cols)} category columns → {X_tr.shape[1]} model inputs")

## Step 7 · Build the network
Two hidden layers (64 and 32 neurons, ReLU) with 20% dropout, and one sigmoid output for yes/no.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)
n_classes = 0 if problem == "regression" else len(labels.classes_)
units, activation, loss, metric = {
    "regression": (1, None, "mse", "mae"),
    "binary": (1, "sigmoid", "binary_crossentropy", "accuracy"),
    "multiclass": (n_classes, "softmax", "sparse_categorical_crossentropy", "accuracy"),
}[problem]

model = keras.Sequential([
    keras.Input(shape=(X_tr.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(units, activation=activation),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss=loss, metrics=[metric])
model.summary()

## Step 8 · Train
Early stopping halts training when the validation loss stops improving, which prevents overfitting.

In [ ]:
import matplotlib.pyplot as plt

stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
BATCH = 32 if len(X_tr) < 5000 else 256
history = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=300, batch_size=BATCH, callbacks=[stop], verbose=0)

h = pd.DataFrame(history.history)
best = h["val_loss"].idxmin()
print(f"Stopped after {len(h)} epochs. Best epoch: {best + 1}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
h[["loss", "val_loss"]].plot(ax=ax[0], title="Loss (lower is better)")
h[[metric, "val_" + metric]].plot(ax=ax[1], title=metric)
for a in ax: a.set_xlabel("epoch")
plt.show()

Training stopped after about 25 epochs and kept the best epoch (around 10 to 15), before the validation loss began to rise.

## Step 9 · Test results
Accuracy on test races the model never saw, compared with always guessing the most common answer.

In [ ]:
from sklearn import metrics as m

if problem == "regression":
    pred = model.predict(X_te, verbose=0).ravel() * y_std + y_mean
    actual = y_test_raw.to_numpy()
    print(f"Average error (MAE): {m.mean_absolute_error(actual, pred):,.2f}")
    print(f"RMSE:                {m.root_mean_squared_error(actual, pred):,.2f}")
    print(f"R² (1.0 is perfect): {m.r2_score(actual, pred):.3f}")
    base = m.mean_absolute_error(actual, np.full_like(actual, y_mean, dtype=float))
    mae, r2 = m.mean_absolute_error(actual, pred), m.r2_score(actual, pred)
    print(f"Baseline MAE (always guess the average): {base:,.2f}\n")
    print(f"On average the model misses by {mae:,.0f}. Always guessing the average misses by {base:,.0f},")
    print(f"   so the model cuts the error by {1 - mae / base:.0%}.")
    print(f"R² = {r2:.2f}: the model explains {max(r2, 0):.0%} of the ups and downs in {TARGET}.")
    plt.figure(figsize=(5, 5))
    plt.scatter(actual, pred, s=6, alpha=0.4)
    lims = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
    plt.plot(lims, lims, "r--")
    plt.xlabel("actual"); plt.ylabel("predicted"); plt.title("Predicted vs actual (closer to the line is better)")
    plt.show()
else:
    proba = model.predict(X_te, verbose=0)
    pred = (proba.ravel() > 0.5).astype(int) if problem == "binary" else proba.argmax(axis=1)
    acc, base = m.accuracy_score(y_te, pred), np.bincount(y_tr).max() / len(y_tr)
    common = labels.classes_[np.bincount(y_tr).argmax()]
    recall = m.recall_score(y_te, pred, average=None, zero_division=0)
    weak = recall.argmin()
    print(f"Test accuracy: {acc:.3f}")
    print(f"Baseline (always guess the most common class): {base:.3f}\n")
    print(f"Right {acc:.0%} of the time. Always guessing '{common}' would be right {base:.0%} of the time,")
    print(f"   so the model is {(acc - base) * 100:+.0f} points better than guessing.")
    print(f"Weakest class: '{labels.classes_[weak]}'. The model catches only {recall[weak]:.0%} of them (recall).\n")
    print(m.classification_report(y_te, pred, target_names=labels.classes_, zero_division=0))
    m.ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=labels.classes_, cmap="Blues", xticks_rotation=45)
    plt.title("Confusion matrix (the diagonal is correct)")
    plt.show()

The model is right about 78% of the time, versus about 67.5% for always guessing "no points" (+10 points). It identifies non-scorers better than scorers: it misses about 4 in 10 real point-scorers, often midfield drivers who gain places from crashes and retirements.

## Step 9b · Comparison with simple rules
The model compared with two rules a fan might use: "starts in the top 10" and "has been finishing in the top 10".

In [ ]:
y_true = y_test_raw.astype(int).to_numpy()
proba_te = model.predict(X_te, verbose=0).ravel()
nn_pred = labels.classes_[(proba_te > 0.5).astype(int)].astype(int)

rules = {
    "Always guess 'no points'": np.zeros_like(y_true),
    "Rule: starts in the top 10": (X_test["grid_pos"] <= 10).astype(int).to_numpy(),
    "Rule: averaged top-10 finishes in last 5 races": (X_test["driver_avg_finish_last5"] <= 10).astype(int).to_numpy(),
    "My neural network": nn_pred,
}
check_table = pd.DataFrame({
    name: {"accuracy": m.accuracy_score(y_true, p),
           "precision (says 'points', is right)": m.precision_score(y_true, p, zero_division=0),
           "recall (catches real point-scorers)": m.recall_score(y_true, p, zero_division=0),
           "F1": m.f1_score(y_true, p, zero_division=0)}
    for name, p in rules.items()}).T.round(3)
print(f"AUC (how well it ranks drivers, 0.5 = coin flip, 1.0 = perfect): {m.roc_auc_score(y_true, proba_te):.3f}")
check_table

## Step 9c · Architecture experiments
Five network designs trained on the same data and ranked by validation accuracy.

In [ ]:
import time

def build(hidden, dropout, lr):
    keras.utils.set_random_seed(42)
    net = keras.Sequential([keras.Input(shape=(X_tr.shape[1],))])
    for i, n in enumerate(hidden):
        net.add(layers.Dense(n, activation="relu"))
        if i == 0 and dropout:
            net.add(layers.Dropout(dropout))
    net.add(layers.Dense(1, activation="sigmoid"))
    net.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
    return net

designs = {
    "A · template: 64 → 32, dropout 0.2": ([64, 32], 0.2, 0.001),
    "B · wider: 128 → 64, dropout 0.2": ([128, 64], 0.2, 0.001),
    "C · deeper: 128 → 64 → 32, dropout 0.3": ([128, 64, 32], 0.3, 0.001),
    "D · smaller: 32 → 16, dropout 0.1": ([32, 16], 0.1, 0.001),
    "E · template, slower learning rate 0.0003": ([64, 32], 0.2, 0.0003),
}
rows = []
for name, (hidden, dropout, lr) in designs.items():
    t0 = time.time()
    net = build(hidden, dropout, lr)
    hist = net.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=300, batch_size=BATCH,
                   callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)], verbose=0)
    p_va = (net.predict(X_va, verbose=0).ravel() > 0.5).astype(int)
    p_te = net.predict(X_te, verbose=0).ravel()
    rows.append({"design": name, "weights": net.count_params(), "epochs": len(hist.history["loss"]),
                 "val accuracy": m.accuracy_score(y_va, p_va), "test accuracy": m.accuracy_score(y_te, (p_te > 0.5).astype(int)),
                 "test AUC": m.roc_auc_score(y_te, p_te), "seconds": round(time.time() - t0, 1)})
results = pd.DataFrame(rows).set_index("design").sort_values("val accuracy", ascending=False).round(3)
print(f"Best design by validation accuracy: {results.index[0]}")
results

All designs scored about 77 to 78%. A bigger network did not help, so the template design was kept. The limit is the information in the data, not the size of the model.

## Step 10 · Spot check
Predictions for three test rows next to the real answers.

In [ ]:
def predict(rows):
    out = model.predict(prep.transform(rows).astype("float32"), verbose=0)
    if problem == "regression":
        return out.ravel() * y_std + y_mean
    idx = (out.ravel() > 0.5).astype(int) if problem == "binary" else out.argmax(axis=1)
    return labels.classes_[idx]

check = X_test.head(3).copy()
check.insert(0, "actual", y_test_raw.head(3).to_numpy())
check.insert(0, "prediction", predict(X_test.head(3)))
check

## Step 11 · Predict a custom row
A front-running driver starting 3rd at Monza, then one value changed at a time.

In [ ]:
typical = {c: (float(X_all[c].median()) if c in num_cols else X_all[c].mode()[0]) for c in X_all.columns}

# A front-running driver in a top team at Monza: starts 3rd, has been finishing around 4th, team won last year's title
MY_VALUES = {"season": 2025, "circuit_id": "monza", "constructor": "McLaren", "grid_pos": 3, "field_size": 20,
             "driver_avg_finish_last5": 4.0, "driver_podium_rate_last10": 0.5,
             "team_avg_finish_last10": 5.0, "team_podium_rate_last10": 0.45,
             "driver_prev_champ_pos": 2, "driver_prev_season_pts": 300, "team_prev_champ_pos": 1}

my_row = pd.DataFrame([{**typical, **MY_VALUES}])
value = predict(my_row)[0]
chance = model.predict(prep.transform(my_row).astype("float32"), verbose=0).ravel()[0]
print("Prediction:", value, "(1 = scores points)", f"· chance of scoring points: {chance:.0%}")
my_row

In [ ]:
# Change one value at a time
def chance_of_points(**changes):
    row = pd.DataFrame([{**typical, **MY_VALUES, **changes}])
    return model.predict(prep.transform(row).astype("float32"), verbose=0).ravel()[0]

print("Starting position (everything else the same):")
for g in [1, 3, 6, 10, 14, 18, 20]:
    print(f"  grid {g:>2}: {chance_of_points(grid_pos=g):5.0%}")
print("\nRecent form, average finish over the last 5 races (starting 10th):")
for f in [3, 6, 10, 14, 18]:
    print(f"  avg finish {f:>2}: {chance_of_points(grid_pos=10, driver_avg_finish_last5=f):5.0%}")

Better grid positions raise the chance of scoring, as expected. Changing recent form alone barely moves it, because the model relies more on team strength and grid position.

## Step 12 · Save
Saves the trained network (`model.keras`) and the preprocessing steps (`prep.joblib`).

In [ ]:
import joblib

model.save("model.keras")
joblib.dump({"prep": prep, "problem": problem, "labels": None if problem == "regression" else labels,
             "y_mean": y_mean if problem == "regression" else None, "y_std": y_std if problem == "regression" else None,
             "columns": list(X_all.columns), "typical": typical},
            "prep.joblib")
print("Saved model.keras and prep.joblib (Colab: open the Files panel on the left to download them)")

## Step 13 · Interactive form
Enter values and click **Predict**.

In [ ]:
import ipywidgets as w
from IPython.display import display

style, layout = {"description_width": "180px"}, w.Layout(width="420px")
inputs = {
    c: w.FloatText(value=round(typical[c], 2), description=c, style=style, layout=layout) if c in num_cols
    else w.Dropdown(options=sorted(X_all[c].dropna().astype(str).unique()), value=str(typical[c]), description=c, style=style, layout=layout)
    for c in X_all.columns
}
button, answer = w.Button(description="Predict", button_style="primary"), w.HTML()

def on_click(_):
    row = pd.DataFrame([{c: box.value for c, box in inputs.items()}])
    value = predict(row)[0]
    answer.value = f"<h3>Prediction: {value:,.2f}</h3>" if problem == "regression" else f"<h3>Prediction: {value}</h3>"

button.on_click(on_click)
display(w.VBox([*inputs.values(), button, answer]))

## Conclusion
- The network predicts points finishes about 78% of the time, beating both the baseline (67.5%) and simple F1 rules (71 to 73%). AUC is about 0.84.
- Most of the work was data preparation: combining the tables and keeping out information only known after the race.
- Limits: no weather, crashes or car upgrades, and the scoring rules changed over the decades.